*   https://unstats.un.org/unsd/snaama
*   https://unstats.un.org/unsd/snaama/Basic (Basic Data Selection → GDP, Annual Rate of Growth - Percentage)

In [ ]:
import pandas as pd

# Load the dataset
file_path = '/content/Results.csv'
df = pd.read_csv(file_path)

# Display the first few rows in a table format
df.head()

,Country/Area,Year,Unit,"GDP, Annual Rate of Growth - Percentage"
0,Afghanistan,1970,...,...
1,Afghanistan,1971,Percentage,-0.2757095194978465
2,Afghanistan,1972,Percentage,-16.401219343449945
3,Afghanistan,1973,Percentage,2.8958465599867367
4,Afghanistan,1974,Percentage,5.453020134203168


In [ ]:
# Assuming columns might be named 'Country', 'Year', and 'GDP Annual Growth Rate (%)' or similar.
# Let's write a flexible parser to find relevant columns.
country_col = [col for col in df.columns if 'country' in col.lower() or 'name' in col.lower()][0]
year_col = [col for col in df.columns if 'year' in col.lower() or 'date' in col.lower()][0]
gdp_col = [col for col in df.columns if 'gdp' in col.lower() and ('growth' in col.lower() or 'rate' in col.lower())][0]

# Drop rows with missing values in essential columns
df_clean = df.dropna(subset=[country_col, year_col, gdp_col]).copy()

# Convert Year and GDP Growth columns to numeric if they aren't already
df_clean[year_col] = pd.to_numeric(df_clean[year_col], errors='coerce')
df_clean[gdp_col] = pd.to_numeric(df_clean[gdp_col], errors='coerce')
df_clean = df_clean.dropna(subset=[year_col, gdp_col])

# Find the most recent year
most_recent_year = int(df_clean[year_col].max())

# Filter for the most recent year
df_recent = df_clean[df_clean[year_col] == most_recent_year]

# Rank countries by GDP growth rate in descending order
ranked_df = df_recent[[country_col, gdp_col]].sort_values(by=gdp_col, ascending=False).reset_index(drop=True)

# Create a structured presentation table with 'Rank'
ranked_table = pd.DataFrame({
    'Rank': range(1, len(ranked_df) + 1),
    'Country/Area': ranked_df[country_col],
    'GDP Annual Growth Rate (%)': ranked_df[gdp_col].round(2)
})

# Display the table cleanly without displaying the default DataFrame index
from IPython.display import HTML
HTML(ranked_table.to_html(index=False))

Rank,Country/Area,GDP Annual Growth Rate (%)
1,Guyana,43.57
2,Palau,12.00
3,Comoros,10.64
4,Georgia,9.68
5,Guinea,9.25
6,Kyrgyzstan,9.04
7,Monaco,8.83
8,Niger,8.48
9,Tajikistan,8.36
10,"China, Macao Special Administrative Region",7.85


In [ ]:
import pandas as pd
from IPython.display import HTML, display

# Ensure we are using the cleaned dataframe with valid country names
available_countries = df_clean[country_col].unique()

selected_country = None

while True:
    search_query = input("Enter a country name (or part of it) to retrieve historical GDP growth (or type 'exit' to cancel): ").strip()
    if not search_query:
        print("Please enter a valid search term.")
        continue

    # Allow the user to exit
    if search_query.lower() in ['exit', 'quit']:
        print("Search cancelled by the user.")
        break

    # Find potential matches (case-insensitive)
    matches = [c for c in available_countries if search_query.lower() in c.lower()]

    if not matches:
        print(f"No countries found matching '{search_query}'. Please try again.\n")
    elif len(matches) == 1:
        selected_country = matches[0]
        print(f"\nShowing full historical data for: {selected_country}")
        break
    else:
        # Display the entire list of matches as a vertical list
        print(f"\nMultiple matches found ({len(matches)}):")
        for m in matches:
            print(f"- {m}")
        print("Please be more specific with your search term.\n")

# Filter and display the selected country's history if a selection was made
if selected_country:
    country_data = df_clean[df_clean[country_col] == selected_country].copy()
    country_data_sorted = country_data.sort_values(by=year_col)

    # Format table for output, including a sequential numbered list
    presentation_df = pd.DataFrame({
        'No.': range(1, len(country_data_sorted) + 1),
        'Year': country_data_sorted[year_col].astype(int),
        'GDP Annual Growth Rate (%)': country_data_sorted[gdp_col].round(2)
    })

    display(HTML(presentation_df.to_html(index=False)))

Enter a country name (or part of it) to retrieve historical GDP growth (or type 'exit' to cancel): exit
Search cancelled by the user.
